In [0]:
bootstrap_servers='pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092'
topic='credit_card_transactions'
# Instead of dbutils.secrets.get("scope", "key")
kafka_api_key = dbutils.secrets.get(
    catalog="finguard_streaming_project",
    schema="bronze",
    key="kafka-api-key"
)
kafka_api_secret = dbutils.secrets.get(
    catalog="finguard_streaming_project",
    schema="bronze",
    key="kafka-api-secret"
)

In [0]:
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{kafka_api_key}" password="{kafka_api_secret}";'

In [0]:
sameple_df = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_servers) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", jaas_config) \
    .option("startingOffsets", "earliest") \
    .option("subscribe", topic) \
    .load()

In [0]:
sameple_df.count()

In [0]:
display(sameple_df);

In [0]:
from pyspark.sql.functions import col
parsed_batch=sameple_df.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)

In [0]:
display(parsed_batch)

In [0]:
parsed_batch.write.saveAsTable("finguard_streaming_project.bronze.transactions_batch_test")

In [0]:
streaming_df = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_servers) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", jaas_config) \
    .option("startingOffsets", "earliest") \
    .option("subscribe", topic) \
    .load()

In [0]:
from pyspark.sql.functions import col
parsed_streaming_df=streaming_df.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)

In [0]:
streaming_query=parsed_streaming_df.writeStream \
    .format("delta") \
    .outputMode("Append") \
    .option("checkpointLocation", "/Volumes/finguard_streaming_project/source/transactions/checkpoint/") \
    .trigger(availableNow=True) \
    .toTable("finguard_streaming_project.bronze.transactions_streaming_test")

print("Query id: ", streaming_query.id)   


In [0]:
%sql
select * from finguard_streaming_project.bronze.transactions_streaming_test

In [0]:
%sql
select * from finguard_streaming_project.bronze.transactions